In [ ]:
import socket
import time
from pynput import keyboard

ROBOT_IP='here should be ur wi-fi ip'
PORT='here should be wi-fi port'

current_key=None

print(f'Connecting to robot at {ROBOT_IP}:{PORT}')

try:
    robot=socket.socket(socket.AF_INET,socket.SOCK_STREAM)
    robot.connect((ROBOT_IP,PORT))
    print('Successfully connected over Wi-Fi!')
    print('\n--- CONTROLS ---')
    print('Use Arrow Keys or WASD to drive the robot.')
    print('Press ESC to exit safely.\n')

    def send_command(command):
        """Helper function to encode and send data over the network."""
        try:
            packet=(command + '\n').encode('utf-8')
            robot.sendall(packet)
            print(f"Sent Network Action: {command}")
        except socket.error as e:
            print(f"\nTransmission failed: {e}")
    def on_press(key):
        global current_key

        if current_key is not None:
            return

        try:
            if key==keyboard.Key.up or getattr(key,'char',None)=='w':
                current_key='FORWARD'
            elif key==keyboard.Key.down or getattr(key,'char',None)=='s':
                current_key='BACKWARD' 
            elif key==keyboard.Key.left or getattr(key,'char',None)=='a':
                current_key='LEFT'
            elif key==keyboard.Key.right or getattr(key,'char',None)=='d':
                current_key='RIGHT'
            elif key==keyboard.Key.esc:
                print("\nExiting controller...")
                return False

            if current_key:
                send_command(current_key)
        except AttributeError:
            pass

    def on_release(key):
        global current_key

        is_arrow_release=key in [keyboard.Key.up,keyboard.Key.down,keyboard.Key.left,keyboard.Key.right]
        is_wasd_release=getattr(key,'char',None) in ['w','a','s','d']

        if is_arrow_release or is_wasd_release:
            current_key=None
            send_command("STOP")
    with keyboard.Listener(on_press=on_press,on_release=on_release) as listener:
        listener.join()

except socket.error as e:
    print(f"Network Connection Failure: {e}")
finally:
    
    try:
        robot.close()
        print("Network Socket Closed cleanly.")
    except NameError:
        pass

Connecting to robot at 192.168.1.45:80
Network Socket Closed cleanly.


KeyboardInterrupt: 